# Fraud Detection
This notebook analyses the source data

## 1 Dataset
**Input file**
-  Parquet file with a dataset of bank tranactions.
  
**Input Flow**:
- The producer extracts a sample of 300 records for a given value of field "step".
- It sends messages by batches of 20 records to the topic "transactions".

## 2. Setup
### 2.1. Pre-requisites
- Sample data available in `data/transactions.parquet`.
- Virtual environment ".venv" activated.
- Kafka cluster of broker servers available.

### 2.2. Python Modules

In [47]:
import os
import pandas as pandas
import matplotlib as plt
import json

### 2.3. Path Configuration

In [ ]:
# Relative paths to data
DATA_PATH = "../data"

## 3. Parquet File

#### 3.1. Get DataFrame
The source file contents are loaded into a Pandas DataFrame

In [17]:
INPUT_PATH = f"{DATA_PATH}/transactions.parquet"

In [4]:
try:
    pdf_fact = pandas.read_parquet(INPUT_PATH)
    nb_extracted_records = len(pdf_fact)
    print(f"Extraction Complete : {nb_extracted_records:,} records read")
except Exception as e:
    print(f"Error reading input file: {e}")

Extraction Complete : 6,362,620 records read


**Input File columns**

The source Parquet file's expected fields are:

| Colonne | Type | Description |
|---------|------|-------------|
| `step` | string (UTC) | Reférence to generated data |
| `type` | string | Type of transaction: debit, credit, transfert, withdrawal, etc. |
| `amount` | string | Amonunt of transaction |
| `name[Orig\|Dest]` | string | Source/Destination account name|
| `[old\|new]balance[Or(i)g\|Dest]` | string | Source/Destination balance before/after transaction |
| `isFraud` | string | 1 if transaction is fraud, 0 otherwise |

### 3.2. Data Exploration

**Data Types**

In [5]:
pandas.DataFrame(pdf_fact.dtypes
                 .rename_axis('field_name'), columns=["data_type"]).T 

field_name,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
data_type,int64,object,float64,object,float64,float64,object,float64,float64,int64


**Data Sample**

In [ ]:
pdf_fact.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0


**Null Values**

No null values were found.

In [7]:
pandas.DataFrame(pdf_fact.isna().sum()
                 .rename_axis('field_name'), columns=["Null_count"]).T 

field_name,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
Null_count,0,0,0,0,0,0,0,0,0,0


**Total Number of frauds**

In [8]:
print( f"{pdf_fact['isFraud'].sum():,} frauds out of {len(pdf_fact):,} transactions")

8,213 frauds out of 6,362,620 transactions


**Categorical Fields**

Field "type"

In [9]:
pandas.DataFrame(pdf_fact["type"].value_counts() \
                 .rename_axis('transaction_type'), columns=["count"]).T \
                 .style.format(thousands=',')

transaction_type,CASH_OUT,PAYMENT,CASH_IN,TRANSFER,DEBIT
count,"2,237,500","2,151,495","1,399,284","532,909","41,432"


Field "step"

743 Steps in the dataset: from 1 to 743.

In [16]:
pandas.DataFrame(pdf_fact["step"]).drop_duplicates().sort_values(by="step") 


,step
0,1
2708,2
3722,3
4274,4
4839,5
...,...
6362560,739
6362570,740
6362576,741
6362598,742


Total Number of accounts

In [71]:
print( f"{pdf_fact['nameOrig'].nunique():,} source accounts")
print( f"{pdf_fact['nameDest'].nunique():,} destination accounts")
print( f"for {len(pdf_fact):,} transactions")

6,353,307 source accounts
2,722,362 destination accounts
for 6,362,620 transactions


### 3.3. Analysis

**Fraud Amount**

In [88]:
amount_total = pdf_fact['amount'].sum()
amount_fraud = pdf_fact[pdf_fact['isFraud'] == 1]['amount'].sum()
amount_wo_fraud = amount_total - amount_fraud
print(f"Fraudulent amount: \t{amount_fraud:20,.0f} $")
print(f"Amount without fraud: \t{amount_wo_fraud:20,.0f} $")
print(f"Total amount: \t\t{amount_total:20,.0f} $")

Fraudulent amount: 	      12,056,415,428 $
Amount without fraud: 	   1,132,336,529,332 $
Total amount: 		   1,144,392,944,760 $


**Fraud Ratio**

In [89]:
fraud_percent = (amount_fraud / amount_total) * 100
print(f"Fraudulent amount percentage: \t{fraud_percent:20,.2f} %")

Fraudulent amount percentage: 	                1.05 %


In [90]:
avg_amount_non_fraud = pdf_fact[pdf_fact['isFraud'] == 0]['amount'].mean()
avg_amount_fraud = pdf_fact[pdf_fact['isFraud'] == 1]['amount'].mean()
print(f"Average amount without fraud: \t{avg_amount_non_fraud:20,.0f} $")
print(f"Average amount with fraud: \t{avg_amount_fraud:20,.0f} $")

Average amount without fraud: 	             178,197 $
Average amount with fraud: 	           1,467,967 $


### 3.4. Conclusion
- The Parquet file contains 6.4 million transactions with non Null values.
- There are 6 transaction types: CASH_OUT, PAYMENT	CASH_IN, TRANSFER, DEBIT
- In average, there is 
   - 1 fraud every 800 transactions
   - 1 transaction per source account 
   - 2 transactions per destination account. 
- There are 743 steps.
- Fraud cost of 1%: $1 billion out of a total of $1 trillion.
- the average amount of fraudulent transactions is 178 K$ = 1 tenth of global average.

## 4. Flow by Producer


### 4.1. Get DataFrame
- the transactions are sent to the Brokers 
- VScode's extension **Kafka Explorer** generated a CSV file with 5K messages 
- the CSV file was loaded into a Pandas DataFrame

In [18]:
INPUT_PATH = f"{DATA_PATH}/kafka-messages-1791390187533.csv"

In [ ]:
try:
    pdf_from_csv = pandas.read_csv(INPUT_PATH)
    nb_extracted_records = len(pdf_from_csv)
    print(f"Extraction Complete : {nb_extracted_records:,} records read")
except Exception as e:
    print(f"Error reading input file: {e}")

Extraction Complete : 5,000 records read


**CSV data fields**

The transaction info is a JSON string in field `value`. 

In [24]:
pdf_fact.dtypes

Timestamp        str
Key          float64
Partition      int64
Offset         int64
Value            str
Headers      float64
dtype: object

**Extract transactions**

In [50]:
display(pdf_fact.head(1))
json.loads(json.loads(pdf_fact.head(1)["Value"].values[0]))

,Timestamp,Key,Partition,Offset,Value,Headers
0,2026-10-07T13:46:39.001Z,NaN,0,13924,"""{\""step\"":75,\""type\"":\""CASH_OUT\"",\""amount\""...",NaN


{'step': 75,
 'type': 'CASH_OUT',
 'amount': 460436.43,
 'nameOrig': 'C568175103',
 'oldbalanceOrg': 460436.43,
 'newbalanceOrig': 0.0,
 'nameDest': 'C283083073',
 'oldbalanceDest': 0.0,
 'newbalanceDest': 460436.43,
 'isFraud': 1}

In [46]:
pdf_transactions = pandas.json_normalize(pdf_fact["Value"].apply(json.loads).apply(json.loads))
pdf_transactions.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,75,CASH_OUT,460436.43,C568175103,460436.43,0.0,C283083073,0.00,460436.43,1
1,75,TRANSFER,460436.43,C1935083121,460436.43,0.0,C83154619,0.00,0.00,1
2,75,CASH_OUT,755813.12,C127671231,755813.12,0.0,C974221574,494575.27,1250388.38,1
3,75,TRANSFER,755813.12,C1153451347,755813.12,0.0,C450761826,0.00,0.00,1
4,75,CASH_OUT,959018.79,C1566688012,959018.79,0.0,C1361631138,0.00,959018.79,1
5,75,TRANSFER,959018.79,C1318698561,959018.79,0.0,C2115087067,0.00,0.00,1
6,75,CASH_OUT,810039.19,C1922445613,810039.19,0.0,C461145537,409035.12,1219074.31,1
7,75,TRANSFER,810039.19,C756454870,810039.19,0.0,C1095120189,0.00,0.00,1
8,74,CASH_OUT,3284197.88,C198379281,3284197.88,0.0,C857746237,200942.08,3485139.96,1
9,74,TRANSFER,3284197.88,C103236252,3284197.88,0.0,C1710384366,0.00,0.00,1


### 4.2. Data Exploration

**Data Types**

In [45]:
pandas.DataFrame(pdf_transactions.dtypes
                 .rename_axis('field_name'), columns=["data_type"]).T 

field_name,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
data_type,int64,str,float64,str,float64,float64,str,float64,float64,int64


**Null Values**

No null values were found.

In [53]:
pandas.DataFrame(pdf_transactions.isna().sum()
                 .rename_axis('field_name'), columns=["Null_count"]).T 

field_name,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
Null_count,0,0,0,0,0,0,0,0,0,0


**Total Number of frauds**

In [54]:
print( f"{pdf_transactions['isFraud'].sum():,} frauds out of {len(pdf_transactions):,} transactions")

348 frauds out of 5,000 transactions


**Categorical Fields**

Field "type"

In [55]:
pandas.DataFrame(pdf_transactions["type"].value_counts() \
                 .rename_axis('transaction_type'), columns=["count"]).T \
                 .style.format(thousands=',')

transaction_type,PAYMENT,CASH_OUT,CASH_IN,TRANSFER,DEBIT
count,"2,059","1,449",918,542,32


Field "step"

743 Steps in the dataset: from 1 to 743.

In [69]:
pdf_transactions[["step"]] \
        .drop_duplicates() \
        .sort_values(by="step") \
        .values.T


array([[34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49,
        50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65,
        66, 67, 68, 69, 70, 71, 72, 73, 74, 75]])

Total Number of accounts

In [67]:
print( f"{pdf_transactions['nameOrig'].nunique():,} source accounts")
print( f"{pdf_transactions['nameDest'].nunique():,} destination accounts")
print( f"for {len(pdf_transactions):,} transactions")

5,000 source accounts
4,956 destination accounts
for 5,000 transactions


### 4.3. Analysis

**Fraudulent Transactions**

In [101]:
pdf_fraud_amount = pdf_transactions[pdf_transactions['isFraud'] == 1]["amount"]
pdf_non_fraud_amount = pdf_transactions[pdf_transactions['isFraud'] == 0]["amount"]
pdf_total_amount = pdf_transactions["amount"]

In [100]:
pdf_fraud_amount

0        460436.43
1        460436.43
2        755813.12
3        755813.12
4        959018.79
           ...    
343      112668.54
597     1021129.67
598     1021129.67
1106     225850.53
1107     225850.53
Name: amount, Length: 348, dtype: float64

**Fraud Amount**

In [102]:
print(f"Fraudulent amount: \t{pdf_fraud_amount.sum():20,.0f} $")
print(f"Amount without fraud: \t{pdf_non_fraud_amount.sum():20,.0f} $")
print(f"Total amount: \t\t{pdf_total_amount.sum():20,.0f} $")

Fraudulent amount: 	         437,079,812 $
Amount without fraud: 	         600,773,303 $
Total amount: 		       1,037,853,114 $


**Fraud Ratio**

In [103]:
fraud_percent = (amount_fraud / amount_total) * 100
print(f"Fraudulent amount percentage: \t{fraud_percent:20,.2f} %")

Fraudulent amount percentage: 	               42.11 %


In [105]:
print(f"Average amount without fraud: \t{pdf_non_fraud_amount.mean():20,.0f} $")
print(f"Average amount with fraud: \t{pdf_fraud_amount.mean():20,.0f} $")

Average amount without fraud: 	             129,143 $
Average amount with fraud: 	           1,255,976 $


#### 3.2.2. Fact Table

In [ ]:
# Store pdf_fact in cache (executes previous steps only once for 2 branches)
pdf_fact.persist()

# Records Count
print_nb_rows = lambda pdf_x, text : print(f"{pdf_x.count():,} rows {text}")
print_nb_rows(pdf_fact, "before transformation")

1,244,203 rows before transformation


#### 3.2.3. Feature Engineering
The expected output fields are grouped according to the key source fields required for their calculation:
* a) Attributes and features per product
* b) Features per session
* c) Features per product per session

A mapping matrix for each group follows, showing the dependence of each feature on the input fields

##### a) Attributes and features per product
<small>

| Output Field \ Input Field | Description | event time | event type | product id | category id | category_code, brand, price | user id | user session |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | 
| **product_id, category_code, brand, price** | Product details | | |X| |X| | |
| **purchased** | True if the product was purchased by the user during his session. | |"purchase"|X| | | |X|
| **num_views _product** | Total views for this product during the session. | |"view"|X| | | |X|

</small>

In [ ]:
# Get identifier "product_id" & product attributes "category_code", "brand" & "price"
pdf_temp = pdf_fact \
    .filter(f.col("event_type").isin(["purchase", "view"])) \
    .select("product_id", "category_code", "brand", "price", "event_type", "user_session") \

# Get feature "purchased"
was_purchased = (f.col("event_type") == "purchase")
f_detect_purchase = f.max(was_purchased.cast("int"))  # max() flags if an event occurred

# Get feature "num_views_product"
was_viewed = (f.col("event_type") == "view")
f_count_views = f.sum(was_viewed.cast("int"))  # sum() counts event occurencies

# Get table "per_product"
pdf_per_product = pdf_temp \
    .groupBy("product_id", "category_code", "brand", "price", "user_session") \
    .agg(
        f_detect_purchase.alias("purchased"),
        f_count_views.alias("num_views_product")
    )

Check results

In [ ]:
print_nb_rows(pdf_fact, "in input table")
print_nb_rows(pdf_per_product, "in table 'Feature per product'")
pdf_per_product.show(3)

1,244,203 rows in input table


837,664 rows in table 'Feature per product'


+----------+--------------------+-----+------+--------------------+---------+-----------------+
|product_id|       category_code|brand| price|        user_session|purchased|num_views_product|
+----------+--------------------+-----+------+--------------------+---------+-----------------+
|   6200547|appliances.enviro...|oasis| 49.81|1865d0ee-1f9b-4c7...|        0|                1|
|   1005115|electronics.smart...|apple|975.57|961ff93c-0af2-431...|        0|                1|
|   1801542|electronics.video.tv|   lg|486.24|22276094-f9ce-4e4...|        0|               13|
+----------+--------------------+-----+------+--------------------+---------+-----------------+
only showing top 3 rows


##### b) Features per session
<small>

| Output Field \ Input Field | Description | event time | event type | product id | category id | category_code, brand, price | user id | user session |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **num_views _session** | Total unique products viewed during the session. | |"view"|X| | | |X|
| **start_time** | Hour and minute the session began. |X| | | | | |X|
| **start_weekday** | Day of the week the session began. |X| | | | | |X|
| **duration** | Total session duration in seconds. |X| | | | | |X|
| **num_prev _sessions** | Count of the user's previous sessions. |X| | | | |X|X|

</small>

**b.1) Aggregated Features**

In [ ]:
# Get feature "num_views_session"
f_count_unique_viewed_products = \
    f.count_distinct(  # count of distinct products viewed per session
        f.when(was_viewed, f.col("product_id"))
    )

# Get temporary feature "temp_start_dt"
f_temp_start_dt = f.min("event_time")

# Get temporary feature "temp_end_dt"
f_temp_end_dt = f.max("event_time")

# Get temporary table "temp_per_session"
pdf_temp = pdf_fact \
    .groupBy("user_session", "user_id") \
    .agg(
        f_count_unique_viewed_products.alias("num_views_session"),
        f_temp_start_dt.alias("temp_start_dt"),
        f_temp_end_dt.alias("temp_end_dt")
    )

Check results

In [ ]:
print_nb_rows(pdf_fact, "in input table")
print_nb_rows(pdf_temp, "in table 'temp per session'")
pdf_temp.show(3)

1,244,203 rows in input table


268,727 rows in table 'temp per session'


+--------------------+---------+-----------------+-------------------+-------------------+
|        user_session|  user_id|num_views_session|      temp_start_dt|        temp_end_dt|
+--------------------+---------+-----------------+-------------------+-------------------+
|727a08f9-c385-443...|537934856|                1|2019-10-01 04:08:55|2019-10-01 04:08:55|
|5ed0d440-f23b-469...|552376713|               17|2019-10-01 06:18:20|2019-10-01 06:43:05|
|48246c5c-cf43-472...|540873367|                4|2019-10-01 06:48:21|2019-10-01 06:48:50|
+--------------------+---------+-----------------+-------------------+-------------------+
only showing top 3 rows


**b.2) Derived & Window-Aggregated Features**

3 derived features from an existing field and 1 window-aggregated.

A **window aggregation** allows for cummulative calculation on previous records.

For example, the count of previous records sorted by time where a target field matches the current record's value.

<small>

    # Step 1. Define the window frame
        period = Window \
            .partitionBy(target_field) \
            .orderBy(time_field) \
            .rowsBetween(lower_bound, upper_bound offset)
    # Step 2. Define a cummulative function applied to the window
        f_cummul = f.count(record_id).over(period)
    # Step 3. Append a column 
        pdf_x = pdf_y.withColumn(new_field, f_cummul)

</small>

In [ ]:
# Get feature "start_time"
f_start_time = f.date_format(f.col("temp_start_dt"), "HH:mm")

# Get feature "start_weekday"
f_start_weekday = f.date_format(f.col("temp_start_dt"), "EEEE")

# Get feature "duration"
f_duration = f.col("temp_end_dt").cast("long") - f.col("temp_start_dt").cast("long")

# Get feature "num_prev_sessions" for a given user
period_before_current_session = Window \
    .partitionBy("user_id") \
    .orderBy("temp_start_dt") \
    .rowsBetween(Window.unboundedPreceding, -1) # excludes current row
f_num_prev_sessions = f.count("user_session").over(period_before_current_session)

# Get table "Features per session"
pdf_per_session = pdf_temp \
    .withColumn("start_time", f_start_time) \
    .withColumn("start_weekday", f_start_weekday) \
    .withColumn("duration", f_duration) \
    .withColumn("num_prev_sessions", f_num_prev_sessions)

Check results

In [ ]:
print_nb_rows(pdf_temp, "in input table")
print_nb_rows(pdf_per_session, "in table 'Features per session'")
pdf_per_session.show(3)

268,727 rows in input table


268,727 rows in table 'Features per session'


+--------------------+---------+-----------------+-------------------+-------------------+----------+-------------+--------+-----------------+
|        user_session|  user_id|num_views_session|      temp_start_dt|        temp_end_dt|start_time|start_weekday|duration|num_prev_sessions|
+--------------------+---------+-----------------+-------------------+-------------------+----------+-------------+--------+-----------------+
|91769fdf-461b-4e4...|244951053|                1|2019-10-01 08:47:35|2019-10-01 08:48:28|     08:47|      Tuesday|      53|                0|
|0051531b-c007-442...|292071852|                1|2019-10-01 17:06:51|2019-10-01 17:06:51|     17:06|      Tuesday|       0|                0|
|d146126f-ce44-48d...|293291933|                4|2019-10-01 18:58:32|2019-10-01 19:03:11|     18:58|      Tuesday|     279|                0|
+--------------------+---------+-----------------+-------------------+-------------------+----------+-------------+--------+-----------------+

##### c) Features per product per session

<small>

| Output Field \ Input Field | Description | event time | event type | product id | category id | category_code, brand, price | user id | user session |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **num_prev _product_views** | Count of previous views for this specific product across past sessions. |X|X|X| | | |X|

</small>

**c.1) Join Table**

The calculation of this field is based on fields from the join of tables "per_product" and "per_session".

In [ ]:
pdf_features = pdf_per_product \
    .join(pdf_per_session, "user_session", "inner")
print_nb_rows(pdf_features, "after join of 2 tables")
pdf_features.show(3)

837,664 rows after join of 2 tables


+--------------------+----------+--------------------+--------+-------+---------+-----------------+---------+-----------------+-------------------+-------------------+----------+-------------+--------+-----------------+
|        user_session|product_id|       category_code|   brand|  price|purchased|num_views_product|  user_id|num_views_session|      temp_start_dt|        temp_end_dt|start_time|start_weekday|duration|num_prev_sessions|
+--------------------+----------+--------------------+--------+-------+---------+-----------------+---------+-----------------+-------------------+-------------------+----------+-------------+--------+-----------------+
|00016bb9-b50d-4bc...|  12300393|construction.tool...|    NULL|   39.1|        0|                2|523769618|                1|2019-10-01 09:51:44|2019-10-01 09:52:14|     09:51|      Tuesday|      30|                0|
|0001c1b8-85f4-46c...|  44500040|                NULL|omabelle|1686.02|        0|                1|512864803|           

**c.2) Aggregated Features**

The feature _`num_prev_product_views`_ represents the total number of views of the current product across all previous sessions but it wasn't clear by whom: the same user or any user. The customer was consulted and he stated that it's by the same user, which makes more sense for UX analysis.

In [ ]:
# Period P of sessions previous to current one
period_before_current_session = Window \
    .partitionBy("product_id", "user_id") \
    .orderBy("temp_start_dt") \
    .rowsBetween(Window.unboundedPreceding, -1)  # omit current row

# Get feature "num_prev_product_views" (# of views of current product during P)
f_num_prev_product_views = f.coalesce(
    f.sum("num_views_product").over(period_before_current_session),
    f.lit(0)  # replace Null by 0
)

# Get table "Features_per_product_per_session"
pdf_features = pdf_features \
    .withColumn(
        "num_prev_product_views",
        f_num_prev_product_views
    )

**Verify Results**

Check counts for the most viewed product by the same user.

- Get Top tuple (user, product)

In [ ]:
pdf_top = pdf_features.groupBy("user_id", "product_id") \
    .count() \
    .sort("count", ascending=False) \
    .limit(1)
pdf_top.show(3)

top_user_id, top_product_id, count = pdf_top.first()

+---------+----------+-----+
|  user_id|product_id|count|
+---------+----------+-----+
|512478069|  28401176|   18|
+---------+----------+-----+



- Check counts for this tuple

In [ ]:
pdf_features \
    .filter(f.col("product_id") == top_product_id) \
    .filter(f.col("user_id") == top_user_id) \
    .select("num_prev_sessions", "num_views_product",
            "num_prev_product_views", "start_time", "temp_start_dt") \
    .show()

+-----------------+-----------------+----------------------+----------+-------------------+
|num_prev_sessions|num_views_product|num_prev_product_views|start_time|      temp_start_dt|
+-----------------+-----------------+----------------------+----------+-------------------+
|                0|                2|                     0|     02:28|2019-10-01 02:28:32|
|                1|                4|                     2|     02:29|2019-10-01 02:29:40|
|                2|                1|                     6|     02:40|2019-10-01 02:40:31|
|                3|                1|                     7|     02:40|2019-10-01 02:40:56|
|                4|                1|                     8|     02:43|2019-10-01 02:43:03|
|                5|                2|                     9|     02:50|2019-10-01 02:50:22|
|                6|                3|                    11|     02:51|2019-10-01 02:51:45|
|                7|                2|                    14|     03:08|2019-10-0

- The values in "num_prev_product_views" are correct according to fields "num_views_product" and "start_time".

#### 3.2.4 Feature Table

Non-expected output fields removal: temporary features and non expected attributes are dropped

In [ ]:
pdf_features = pdf_features.drop("temp_start_dt", "temp_end_dt") #, "user_id")

The Feature Table contains the 13 expected output fields. 

In [ ]:
print_nb_rows(pdf_features, "in 'Feature Table'")
pdf_features.show(3)
print("***** Transformation complete *****")

837,664 rows in 'Feature Table'


+--------------------+----------+--------------------+-----+------+---------+-----------------+---------+-----------------+----------+-------------+--------+-----------------+----------------------+
|        user_session|product_id|       category_code|brand| price|purchased|num_views_product|  user_id|num_views_session|start_time|start_weekday|duration|num_prev_sessions|num_prev_product_views|
+--------------------+----------+--------------------+-----+------+---------+-----------------+---------+-----------------+----------+-------------+--------+-----------------+----------------------+
|4a3656f4-47d8-4d5...|   1001588|electronics.smart...|meizu| 128.3|        1|                4|513568790|                2|     15:57|      Tuesday|    1366|                0|                     0|
|a521435c-5869-4a6...|   1001588|electronics.smart...|meizu| 128.3|        0|                1|513568790|                5|     20:05|      Tuesday|     503|                1|                     4|
|a287

### 3.3. Load
Writing the "Feature Table" to the output folder in CSV format.

In [ ]:
# Force output to only 1 file with .coalesce()
# Replace existing data in 'overwrite' mode
pdf_features \
    .coalesce(1) \
    .write.mode("overwrite") \
    .csv(OUTPUT_PATH, header=True)

print(f"Data successfully exported to: {OUTPUT_PATH}")

Data successfully exported to: ../data/processed/prototype_output


Check files loading

In [100]:
# Basic check of folder contents
if os.path.exists(OUTPUT_PATH):
    print("Output folder exists.")
    files = os.listdir(OUTPUT_PATH)
    print(f"Generated files:\n-  {"\n-  ".join(files)}")
else:
    print("Error: Output folder was not created.")

Output folder exists.
Generated files:
-  part-00000-71ed6eea-e27c-48a1-a2fe-a03f7a1a6987-c000.csv
-  ._SUCCESS.crc
-  _SUCCESS
-  .part-00000-71ed6eea-e27c-48a1-a2fe-a03f7a1a6987-c000.csv.crc


## 4. End Spark session

In [101]:
spark.stop()

## 5. Annex : Memory Storage
- Get local names of Spark DataFrames cached in cluster's memory.

In [ ]:
cached_pdf_names = [key for key, val in globals().items() \
                    if isinstance(val, DataFrame) \
                        and key != "DataFrame" \
                        and val.is_cached
                    ]

print(f"Cached Spark DataFrames in cluster memory: {cached_pdf_names}")

Cached Spark DataFrames in cluster memory: ['sdf_fact']


- Check Spark DataFrames stored as JVM Spark RDD's in Cluster's memory. 
  
Cf. http://localhost:4040/storage/

<img src="../docs/Local_Spark_Storage_for_nb_prototype.png" width="50%">